In [ ]:
# Intall Required Libraries

!pip install tensorflow numpy PyPDF2 -q

import numpy as np
import PyPDF2
import tensorflow as tf

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense, Bidirectional
from tensorflow.keras.losses import SparseCategoricalCrossentropy


In [ ]:
# Read Text From PDF

pdf_path = "/content/Project Gutenberg.pdf"

text = ""

with open(pdf_path, "rb") as file:
    reader = PyPDF2.PdfReader(file)

    for page in reader.pages[:40]:   # Limited Pages to Reduce Memory
        extracted_text = page.extract_text()

        if extracted_text:
            text += extracted_text + " "


In [ ]:
# Tokenization

tokenizer = Tokenizer(num_words=5000)  # Limit Vocab Size
tokenizer.fit_on_texts([text])

total_words = min(5000, len(tokenizer.word_index) + 1)

print("Total Words:", total_words)


Total Words: 2212


In [ ]:
# CREATE INPUT SEQUENCES


input_sequences = []

for line in text.split('.'):

    token_list = tokenizer.texts_to_sequences([line])[0]

    for i in range(1, len(token_list)):
        n_gram_sequence = token_list[:i+1]
        input_sequences.append(n_gram_sequence)


In [ ]:
# PADDING

max_sequence_len = min(
    max(len(x) for x in input_sequences),
    20
)

input_sequences = np.array(
    pad_sequences(
        input_sequences,
        maxlen=max_sequence_len,
        padding='pre'
    )
)


In [ ]:
# SPLIT FEATURES AND LABELS


X = input_sequences[:, :-1]
y = input_sequences[:, -1]

# IMPORTANT:
# DO NOT USE to_categorical()
# This prevents RAM crash


In [ ]:
# RNN MODEL


rnn_model = Sequential([
    Embedding(total_words, 64, input_length=max_sequence_len-1),
    SimpleRNN(64),
    Dense(total_words, activation='softmax')
])

rnn_model.compile(
    loss=SparseCategoricalCrossentropy(),
    optimizer='adam',
    metrics=['accuracy']
)

print("\nTraining RNN Model...\n")

rnn_model.fit(
    X,
    y,
    epochs=20,
    batch_size=64,
    verbose=1
)



Training RNN Model...

Epoch 1/20
135/135 ━━━━━━━━━━━━━━━━━━━━ 4s 15ms/step - accuracy: 0.0469 - loss: 6.7752
Epoch 2/20
135/135 ━━━━━━━━━━━━━━━━━━━━ 2s 15ms/step - accuracy: 0.0499 - loss: 6.2983
Epoch 3/20
135/135 ━━━━━━━━━━━━━━━━━━━━ 2s 15ms/step - accuracy: 0.0579 - loss: 6.1279
Epoch 4/20
135/135 ━━━━━━━━━━━━━━━━━━━━ 2s 15ms/step - accuracy: 0.0695 - loss: 5.9724
Epoch 5/20
135/135 ━━━━━━━━━━━━━━━━━━━━ 3s 25ms/step - accuracy: 0.0731 - loss: 5.8028
Epoch 6/20
135/135 ━━━━━━━━━━━━━━━━━━━━ 2s 16ms/step - accuracy: 0.0815 - loss: 5.6386
Epoch 7/20
135/135 ━━━━━━━━━━━━━━━━━━━━ 2s 15ms/step - accuracy: 0.0937 - loss: 5.4631
Epoch 8/20
135/135 ━━━━━━━━━━━━━━━━━━━━ 2s 15ms/step - accuracy: 0.1088 - loss: 5.2934
Epoch 9/20
135/135 ━━━━━━━━━━━━━━━━━━━━ 2s 15ms/step - accuracy: 0.1202 - loss: 5.1279
Epoch 10/20
135/135 ━━━━━━━━━━━━━━━━━━━━ 2s 15ms/step - accuracy: 0.1302 - loss: 4.9731
Epoch 11/20
135/135 ━━━━━━━━━━━━━━━━━━━━ 4s 27ms/step - accuracy: 0.1378 - loss: 4.8202
Epoch 12/20
135/1

In [ ]:
# BI-RNN MODEL


birnn_model = Sequential([
    Embedding(total_words, 64, input_length=max_sequence_len-1),
    Bidirectional(SimpleRNN(64)),
    Dense(total_words, activation='softmax')
])

birnn_model.compile(
    loss=SparseCategoricalCrossentropy(),
    optimizer='adam',
    metrics=['accuracy']
)

print("\nTraining BI-RNN Model...\n")

birnn_model.fit(
    X,
    y,
    epochs=20,
    batch_size=64,
    verbose=1
)



Training BI-RNN Model...

Epoch 1/20
135/135 ━━━━━━━━━━━━━━━━━━━━ 6s 22ms/step - accuracy: 0.0437 - loss: 6.7205
Epoch 2/20
135/135 ━━━━━━━━━━━━━━━━━━━━ 3s 22ms/step - accuracy: 0.0508 - loss: 6.2670
Epoch 3/20
135/135 ━━━━━━━━━━━━━━━━━━━━ 3s 24ms/step - accuracy: 0.0633 - loss: 6.0454
Epoch 4/20
135/135 ━━━━━━━━━━━━━━━━━━━━ 4s 30ms/step - accuracy: 0.0794 - loss: 5.7292
Epoch 5/20
135/135 ━━━━━━━━━━━━━━━━━━━━ 3s 22ms/step - accuracy: 0.0961 - loss: 5.4683
Epoch 6/20
135/135 ━━━━━━━━━━━━━━━━━━━━ 3s 22ms/step - accuracy: 0.1146 - loss: 5.1721
Epoch 7/20
135/135 ━━━━━━━━━━━━━━━━━━━━ 3s 25ms/step - accuracy: 0.1266 - loss: 4.9133
Epoch 8/20
135/135 ━━━━━━━━━━━━━━━━━━━━ 4s 30ms/step - accuracy: 0.1420 - loss: 4.6950
Epoch 9/20
135/135 ━━━━━━━━━━━━━━━━━━━━ 3s 22ms/step - accuracy: 0.1610 - loss: 4.4454
Epoch 10/20
135/135 ━━━━━━━━━━━━━━━━━━━━ 5s 22ms/step - accuracy: 0.1801 - loss: 4.2353
Epoch 11/20
135/135 ━━━━━━━━━━━━━━━━━━━━ 4s 33ms/step - accuracy: 0.1994 - loss: 4.0233
Epoch 12/20
13

In [ ]:
# NEXT WORD PREDICTION


def predict_next_word(model, text_input):

    token_list = tokenizer.texts_to_sequences([text_input])[0]

    token_list = pad_sequences(
        [token_list],
        maxlen=max_sequence_len-1,
        padding='pre'
    )

    predicted = model.predict(token_list, verbose=0)

    predicted_word_index = np.argmax(predicted)

    output_word = ""

    for word, index in tokenizer.word_index.items():
        if index == predicted_word_index:
            output_word = word
            break

    return output_word

In [ ]:
# TESTING


seed_text = " But for the trained reasoner to admit such intrusions into his "

print("\nUsing RNN:")
print("Predicted Word:",
      predict_next_word(rnn_model, seed_text))

print("\nUsing BI-RNN:")
print("Predicted Word:",
      predict_next_word(birnn_model, seed_text))


Using RNN:
Predicted Word: hand

Using BI-RNN:
Predicted Word: own
